In [ ]:
"""
================================================================================
OGGM GLACIER ANALYSIS - HISTORICAL Simulation + FUTURE Projections

# ---------------------------------------------------------------------------
# 1: Imports + OGGM Hub setup
# ---------------------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import oggm.cfg as cfg
from oggm import utils, workflow, tasks, DEFAULT_BASE_URL
from oggm.shop import gcm_climate

# ===========================================================================
# GLACIER REGISTRY - EDIT ENTRIES HERE TO SWITCH GLACIERS
# ===========================================================================
GLACIERS = {
    'Guanaco': {
        'rgi_ids': [
            'RGI60-17.14868',
            'RGI60-17.14869',
        ],
        'label':          'Guanaco glacier',
        'rgi_area_km2':   None,
        'rgi_year':       None,
        'obs_area_rate':  None,
        'obs_mb':         None,
    },
    'Coropuna': {
        'rgi_ids': [
            'RGI60-16.01473', 'RGI60-16.01474', 'RGI60-16.01475',
            'RGI60-16.01476', 'RGI60-16.01477', 'RGI60-16.01478',
            'RGI60-16.01479', 'RGI60-16.01480', 'RGI60-16.01481',
            'RGI60-16.01482', 'RGI60-16.01483', 'RGI60-16.01484',
            'RGI60-16.01485', 'RGI60-16.01486', 'RGI60-16.01487',
            'RGI60-16.01488', 'RGI60-16.01489', 'RGI60-16.01490',
            'RGI60-16.01491', 'RGI60-16.01492', 'RGI60-16.01493',
            'RGI60-16.01494', 'RGI60-16.01607', 'RGI60-16.01608',
            'RGI60-16.01609', 'RGI60-16.01613',
        ],
        'label':          'Coropuna icefield',
        'rgi_area_km2':   48.24,
        'rgi_year':       2009,
        'obs_area_rate':  0.37,
        'obs_mb':         -0.36,
    },
    'Osorno': {
        'rgi_ids':       ['RGI60-17.12386',   # 2.555 km²
    'RGI60-17.12388',   # 0.268 km²
    'RGI60-17.12390',   # 1.263 km²
    'RGI60-17.12392',   # 0.363 km²
    'RGI60-17.12394',],
        'label':          'Osorno volcano icecap',
        'rgi_area_km2':   None,
        'rgi_year':       None,
        'obs_area_rate':  None,
        'obs_area_note':  '',
        'obs_mb':         None,
        'obs_mb_note':    '',
    },
}

# === CHANGE THIS LINE TO SWITCH GLACIERS ===================================
GLACIER_NAME = 'Guanaco'
# ===========================================================================

glacier  = GLACIERS[GLACIER_NAME]
rgi_ids  = glacier['rgi_ids']
label    = glacier['label']

if not rgi_ids:
    raise ValueError(f"GLACIERS['{GLACIER_NAME}']['rgi_ids'] is empty. "
                     f"Add RGI IDs before running.")

# Historical validation window 
YS = 2000
YE = 2019

# Runoff analysis window 
RUNOFF_YS = 2021
RUNOFF_YE = 2099

# ---------------------------------------------------------------------------
# 2: OGGM initialisation + per-glacier output folder
# ---------------------------------------------------------------------------
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir(
    f'OGGM_{GLACIER_NAME}_run', reset=True)
cfg.PARAMS['min_ice_thick_for_length'] = 1
cfg.PARAMS['store_model_geometry']     = True
cfg.PARAMS['continue_on_error']        = True

OUTDIR = os.path.join(os.getcwd(), f'{GLACIER_NAME}_outputs')
os.makedirs(OUTDIR, exist_ok=True)
print(f'Output folder for downloadable files:\n   {OUTDIR}')

gdirs = workflow.init_glacier_directories(
    rgi_ids,
    from_prepro_level=5,
    prepro_base_url=DEFAULT_BASE_URL,
)
print(f'\nLoaded {len(gdirs)} glacier directories for {label}:')
for g in gdirs:
    print(f'  {g.rgi_id}:  area = {g.rgi_area_km2:.3f} km^2   '
          f'rgi_date = {g.rgi_date}')

if glacier['rgi_area_km2'] is None:
    glacier['rgi_area_km2'] = sum(g.rgi_area_km2 for g in gdirs)
if glacier['rgi_year'] is None:
    glacier['rgi_year'] = gdirs[0].rgi_date

# ---------------------------------------------------------------------------
# 3: CMIP6 GCM download + bias correction + projection runs
# ---------------------------------------------------------------------------
GCMS = [
    'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'EC-Earth3-Veg', 'FGOALS-f3-L',
    'GFDL-ESM4', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-MM',
]
SSPS = ['ssp126', 'ssp245', 'ssp370', 'ssp585']

bp_tmpl = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
           '{gcm}/{gcm}_{ssp}_r1i1p1f1_pr.nc')
bt_tmpl = ('https://cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/'
           '{gcm}/{gcm}_{ssp}_r1i1p1f1_tas.nc')

for gcm in GCMS:
    for ssp in SSPS:
        fid = f'_CMIP6_{gcm}_{ssp}'
        print(f'\n>>> Processing {gcm} {ssp}')
        try:
            ft = utils.file_downloader(bt_tmpl.format(gcm=gcm, ssp=ssp))
            fp = utils.file_downloader(bp_tmpl.format(gcm=gcm, ssp=ssp))
            if ft is None or fp is None:
                print('   skipped (file missing on server)')
                continue
            workflow.execute_entity_task(
                gcm_climate.process_cmip_data, gdirs,
                filesuffix=fid,
                fpath_temp=ft,
                fpath_precip=fp,
            )
            workflow.execute_entity_task(
                tasks.run_with_hydro, gdirs,
                run_task=tasks.run_from_climate_data,
                climate_filename='gcm_data',
                climate_input_filesuffix=fid,
                init_model_filesuffix='_spinup_historical',
                output_filesuffix=fid,
                store_monthly_hydro=False,
            )
        except Exception as e:
            print(f'   FAILED: {e}')

# ---------------------------------------------------------------------------
# 4: Compile projection runs into one Dataset (time x rgi_id x gcm x ssp)
# ---------------------------------------------------------------------------
def compile_all():
    all_ds = []
    for gcm in GCMS:
        for ssp in SSPS:
            fid = f'_CMIP6_{gcm}_{ssp}'
            try:
                ds = utils.compile_run_output(gdirs, input_filesuffix=fid)
            except Exception as e:
                print(f'skip {gcm} {ssp}: {e}')
                continue
            ds.attrs = {}
            for var in ds.data_vars:
                ds[var].attrs = {}
            ds = ds.expand_dims({'gcm': [gcm], 'ssp': [ssp]})
            all_ds.append(ds)
    return xr.combine_by_coords(all_ds, combine_attrs='override')

DS = compile_all()
DS = DS.sel(time=slice(2020, 2100))
print('\nCompiled projection dataset dims:', dict(DS.sizes))
DS.to_netcdf(os.path.join(OUTDIR, f'{GLACIER_NAME}_all_runs.nc'))

# ---------------------------------------------------------------------------
# 5: Compile the _spinup_historical run + run dedicated historical hydro
# ---------------------------------------------------------------------------
DS_HIST = utils.compile_run_output(gdirs, input_filesuffix='_spinup_historical')

HYDRO_SUFFIX = f'_hydro_{YS}_{YE}'
print(f'\n>>> Running historical run_with_hydro {YS}-{YE}')
try:
    workflow.execute_entity_task(
        tasks.run_with_hydro, gdirs,
        run_task=tasks.run_from_climate_data,
        ys=YS,
        ye=YE,
        init_model_filesuffix='_spinup_historical',
        init_model_yr=YS,
        ref_area_from_y0=True,
        output_filesuffix=HYDRO_SUFFIX,
        store_monthly_hydro=True,
    )
    HYDRO_OK = True
    print('   done.')
except Exception as e:
    HYDRO_OK = False
    print(f'   FAILED (historical runoff summary will be skipped): {e}')

# ---------------------------------------------------------------------------
# 6: Constants + plotting 
# ---------------------------------------------------------------------------
ICE_DENSITY = 0.850   

SSP_COLOURS = {
    'ssp126': '#1f77b4', 'ssp245': '#2ca02c',
    'ssp370': '#ff7f0e', 'ssp585': '#d62728',
}
SSP_LABELS = {
    'ssp126': 'SSP1-2.6', 'ssp245': 'SSP2-4.5',
    'ssp370': 'SSP3-7.0', 'ssp585': 'SSP5-8.5',
}

ANNUAL_C = '#1f4e79'
MEAN5_C  = 'green'
MEAN11_C = 'orange'
OBS_C    = 'red'

def roll(vals, w):
    return pd.Series(vals).rolling(window=w, center=True,
                                   min_periods=1).mean().values

def ensemble_mean_std(var, sum_glaciers=True):
    out = {}
    for ssp in SSPS:
        da = DS[var].sel(ssp=ssp)
        if sum_glaciers and 'rgi_id' in da.dims:
            da = da.sum('rgi_id')
        out[ssp] = pd.DataFrame({
            'time': da.time.values,
            'mean': da.mean('gcm').values,
            'std':  da.std('gcm').values,
        })
    return out

def styled_plot(ax, df, ssp, col_mean='mean', col_std='std'):
    c = SSP_COLOURS[ssp]
    ax.fill_between(df['time'], df[col_mean]-df[col_std],
                    df[col_mean]+df[col_std], color=c, alpha=0.15)
    ax.plot(df['time'], df[col_mean], color=c, lw=2, label=SSP_LABELS[ssp])

def get_years(time_coord):
    try:
        return time_coord.dt.year.values.tolist()
    except AttributeError:
        return [int(str(t)[:4]) for t in time_coord.values]

# ===========================================================================
# 7: HISTORICAL ANALYSIS (2000-2019)
# ===========================================================================
all_years = get_years(DS_HIST.time)
keep      = [i for i, y in enumerate(all_years) if YS <= y <= YE]
ds_hist   = DS_HIST.isel(time=keep)

volume = ds_hist.volume.sum(dim='rgi_id')
area   = ds_hist.area.sum(dim='rgi_id')

years_av  = get_years(volume.time)
vol_vals  = volume.values
area_vals = area.values
area_km2  = area_vals / 1e6

area_loss_cum = area_km2[0] - area_km2
area_pct_loss = (area_loss_cum / area_km2[0]) * 100
vol_pct       = (vol_vals / vol_vals[0]) * 100
vol_loss_pct  = 100 - vol_pct[-1]

area_loss_total = area_km2[0] - area_km2[-1]

coeffs     = np.polyfit(years_av, area_km2, 1)
area_rate  = coeffs[0]
area_trend = np.polyval(coeffs, years_av)

area_mean    = 0.5 * (area_vals[:-1] + area_vals[1:])
mass_balance = (np.diff(vol_vals) * ICE_DENSITY) / area_mean
years_mb     = years_av[1:]

GLACIER_RUNOFF_VARS = ['melt_on_glacier',  'liq_prcp_on_glacier']
ALL_RUNOFF_VARS     = GLACIER_RUNOFF_VARS + [
                       'melt_off_glacier', 'liq_prcp_off_glacier']

gr_vals      = None
tr_vals      = None
years_runoff = None
if HYDRO_OK:
    glacier_runoff_sum = None
    total_runoff_sum   = None
    for gd in gdirs:
        hp = gd.get_filepath('model_diagnostics', filesuffix=HYDRO_SUFFIX)
        if not os.path.exists(hp):
            print(f'  WARNING: no hydro file for {gd.rgi_id}')
            continue
        with xr.open_dataset(hp) as ds_h:
            ds_h = ds_h.isel(time=slice(0, -1)).load()
        g_r = (ds_h[GLACIER_RUNOFF_VARS].to_array(dim='c')
               .sum('c').clip(min=0) * 1e-9)
        t_r = (ds_h[ALL_RUNOFF_VARS].to_array(dim='c')
               .sum('c').clip(min=0) * 1e-9)
        glacier_runoff_sum = (g_r if glacier_runoff_sum is None
                              else glacier_runoff_sum + g_r)
        total_runoff_sum   = (t_r if total_runoff_sum   is None
                              else total_runoff_sum   + t_r)
    if glacier_runoff_sum is not None:
        gr_vals      = glacier_runoff_sum.values
        tr_vals      = total_runoff_sum.values
        years_runoff = get_years(glacier_runoff_sum.time)

area_5,  area_11  = roll(area_km2,      5), roll(area_km2,      11)
loss_5,  loss_11  = roll(area_loss_cum, 5), roll(area_loss_cum, 11)
vpct_5,  vpct_11  = roll(vol_pct,       5), roll(vol_pct,       11)
mb_5,    mb_11    = roll(mass_balance,  5), roll(mass_balance,  11)
if gr_vals is not None:
    gr_5, gr_11 = roll(gr_vals, 5), roll(gr_vals, 11)
    tr_5, tr_11 = roll(tr_vals, 5), roll(tr_vals, 11)

# ---------------------------------------------------------------------------
# 7a: Hhistorical figure
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

def _fmt(ax, x, tick_step=2):
    ax.set_xlim(min(x) - 0.5, max(x) + 0.5)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_step))
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Year')

def _panel(ax, x, y, y5, y11, title, ylabel,
           hline=None, obs_val=None, obs_label=None, tick_step=2):
    ax.plot(x, y,   color=ANNUAL_C, lw=1.2, label='Annual')
    ax.plot(x, y5,  color=MEAN5_C,  lw=1.5, label='5-yr mean')
    ax.plot(x, y11, color=MEAN11_C, lw=2.5, label='11-yr mean')
    if hline is not None:
        ax.axhline(hline, color='grey', ls='--', lw=1)
    if obs_val is not None:
        ax.axhline(obs_val, color=OBS_C, ls=':', lw=1.8, label=obs_label)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    _fmt(ax, x, tick_step)

ax = axes[0]
ax.plot(years_av, area_km2,   color=ANNUAL_C, lw=1.2, label='Annual')
ax.plot(years_av, area_5,     color=MEAN5_C,  lw=1.5, label='5-yr mean')
ax.plot(years_av, area_11,    color=MEAN11_C, lw=2.5, label='11-yr mean')
ax.plot(years_av, area_trend, color=OBS_C,    lw=1.5, ls='--',
        label=f'Linear trend: {area_rate:.3f} km$^2$ a$^{{-1}}$')
ax.set_title(f'{label} - Glacier area {YS}-{YE}',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Area (km$^2$)')
ax.legend(fontsize=9)
_fmt(ax, years_av)

_panel(axes[1], years_av, area_loss_cum, loss_5, loss_11,
       f'{label} - Cumulative area loss {YS}-{YE}',
       f'Area lost from {YS} (km$^2$)', hline=0)

_panel(axes[2], years_av, vol_pct, vpct_5, vpct_11,
       f'{label} - Relative volume change {YS}-{YE}',
       f'Volume (% of {YS})', hline=100)

mb_obs_kwargs = {}
if glacier['obs_mb'] is not None:
    mb_obs_kwargs = {
        'obs_val':   glacier['obs_mb'],
        'obs_label': f'Observed: {glacier["obs_mb"]} m w.e. a$^{{-1}}$',
    }
_panel(axes[3], years_mb, mass_balance, mb_5, mb_11,
       f'{label} - Area-averaged mass balance {YS+1}-{YE}',
       'Mass balance (m w.e. a$^{-1}$)', hline=0, **mb_obs_kwargs)

if gr_vals is not None:
    _panel(axes[4], years_runoff, gr_vals, gr_5, gr_11,
           f'{label} - Glacier runoff {YS}-{YE}',
           'Runoff (Mt)')
    _panel(axes[5], years_runoff, tr_vals, tr_5, tr_11,
           f'{label} - Total catchment runoff {YS}-{YE}',
           'Runoff (Mt)')
else:
    for ax in (axes[4], axes[5]):
        ax.text(0.5, 0.5, 'Historical hydro run failed -\nrunoff unavailable',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=11, color='grey')
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
hist_fig_path = os.path.join(
    OUTDIR, f'{GLACIER_NAME}_historical_{YS}_{YE}.png')
fig.savefig(hist_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Historical figure saved: {hist_fig_path}')

# ---------------------------------------------------------------------------
# 7b: Historical CSV exports
# ---------------------------------------------------------------------------
pd.DataFrame({
    'year':                    years_av,
    'area_km2':                area_km2,
    'area_5yr_km2':            area_5,
    'area_11yr_km2':           area_11,
    'area_trend_km2':          area_trend,
    f'area_loss_from_{YS}_km2':area_loss_cum,
    'area_loss_5yr_km2':       loss_5,
    'area_loss_11yr_km2':      loss_11,
    f'area_pct_lost_from_{YS}':area_pct_loss,
    'volume_m3':               vol_vals,
    f'volume_pct_of_{YS}':     vol_pct,
    'volume_pct_5yr':          vpct_5,
    'volume_pct_11yr':         vpct_11,
}).to_csv(
    os.path.join(OUTDIR, f'{GLACIER_NAME}_area_volume_{YS}_{YE}.csv'),
    index=False)

pd.DataFrame({
    'year':              years_mb,
    'mass_balance_mwea': mass_balance,
    'mass_balance_5yr':  mb_5,
    'mass_balance_11yr': mb_11,
}).to_csv(
    os.path.join(OUTDIR, f'{GLACIER_NAME}_mass_balance_{YS+1}_{YE}.csv'),
    index=False)

if gr_vals is not None:
    pd.DataFrame({
        'year':                years_runoff,
        'glacier_runoff_Mt':   gr_vals,
        'glacier_runoff_5yr':  gr_5,
        'glacier_runoff_11yr': gr_11,
        'total_runoff_Mt':     tr_vals,
        'total_runoff_5yr':    tr_5,
        'total_runoff_11yr':   tr_11,
    }).to_csv(
        os.path.join(OUTDIR, f'{GLACIER_NAME}_runoff_{YS}_{YE}.csv'),
        index=False)

# ===========================================================================
# 8: FUTURE PROJECTION FIGURES
# ===========================================================================

# --- 8a: Volume change (relative %, 2020 = 100) ----------------------------
vol_ens = ensemble_mean_std('volume', sum_glaciers=True)

fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    df = vol_ens[ssp].copy()
    ref = df['mean'].iloc[0]
    df['mean'] = df['mean'] / ref * 100.
    df['std']  = df['std']  / ref * 100.
    styled_plot(ax, df, ssp)
    end_m, end_s = df['mean'].iloc[-1], df['std'].iloc[-1]
    ax.text(2101, end_m, f'{end_m:.0f} ± {end_s:.0f}%',
            color=SSP_COLOURS[ssp], va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Volume (% of 2020)')
ax.set_title(f'{label} - projected volume change (2020-2100)')
ax.legend(fontsize=10, loc='lower left')
ax.grid(alpha=0.3)
ax.set_xlim(2020, 2105)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_volume_change.png'), dpi=200)

# --- 8b: Area change (km^2) ------------------------------------------------
area_ens = ensemble_mean_std('area', sum_glaciers=True)

fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    df = area_ens[ssp].copy()
    df['mean'] *= 1e-6
    df['std']  *= 1e-6
    styled_plot(ax, df, ssp)
ax.set_xlabel('Year')
ax.set_ylabel('Glacier area (km$^{2}$)')
ax.set_title(f'{label} - projected area change (2020-2100)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_area_change.png'), dpi=200)

# --- 8c: Area-averaged annual mass balance ---------------------
A_RGI_TOT_M2 = sum(g.rgi_area_km2 for g in gdirs) * 1e6

fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    da_vol = DS['volume'].sel(ssp=ssp).sum('rgi_id')
    dv = da_vol.diff('time')
    B  = dv * ICE_DENSITY / A_RGI_TOT_M2
    df = pd.DataFrame({
        'time': B.time.values,
        'mean': B.mean('gcm').values,
        'std':  B.std('gcm').values,
    })
    styled_plot(ax, df, ssp)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Area-averaged mass balance B (m w.e. yr$^{-1}$)')
ax.set_title(f'{label} - area-averaged annual mass balance (2021-2100)  '
             r'($B = \Delta V \times 0.850 / A_{RGI}$, Huss 2013)')
ax.legend(fontsize=10, loc='lower left')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_mass_balance.png'), dpi=200)

# --- 8d: Annual runoff (Mt yr-1) -------

RUNOFF_VARS = ['melt_on_glacier', 'melt_off_glacier',
               'liq_prcp_on_glacier', 'liq_prcp_off_glacier']
runoff = sum(DS[v] for v in RUNOFF_VARS if v in DS)
runoff_trim = runoff.sel(time=slice(RUNOFF_YS, RUNOFF_YE))

fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    da = runoff_trim.sel(ssp=ssp).sum('rgi_id') * 1e-9
    df = pd.DataFrame({
        'time': da.time.values,
        'mean': da.mean('gcm').values,
        'std':  da.std('gcm').values,
    })
    styled_plot(ax, df, ssp)
ax.set_xlabel('Year')
ax.set_ylabel('Total annual runoff (Mt yr$^{-1}$)')
ax.set_title(f'{label} - projected annual runoff '
             f'({RUNOFF_YS}-{RUNOFF_YE})\n'
             '(on- + off-glacier melt + liquid precipitation)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_runoff.png'), dpi=200)

# --- 8e: Mean ice thickness (V / A) ----------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
for ssp in SSPS:
    v = DS['volume'].sel(ssp=ssp).sum('rgi_id')
    a = DS['area'].sel(ssp=ssp).sum('rgi_id')
    h = v / a.where(a > 0)
    df = pd.DataFrame({
        'time': h.time.values,
        'mean': h.mean('gcm').values,
        'std':  h.std('gcm').values,
    })
    styled_plot(ax, df, ssp)
ax.set_xlabel('Year')
ax.set_ylabel('Mean ice thickness V/A (m)')
ax.set_title(f'{label} - projected mean ice thickness (2020-2100)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, 'fig_thickness.png'), dpi=200)

plt.show()

# ===========================================================================
# 9: HISTORICAL VALIDATION 
# ===========================================================================
print(f"\n=== {label.upper()} {YS}-{YE} ===")

print(f"\n  AREA:")
print(f"    {YS}: {area_km2[0]:.2f} km^2")
print(f"      [RGI 6.0 ({glacier['rgi_year']}): "
      f"{glacier['rgi_area_km2']:.2f} km^2]")
print(f"    {YE}: {area_km2[-1]:.2f} km^2")
print(f"    Total loss {YS}-{YE}: {area_loss_total:.2f} km^2")
print(f"    OLS rate: {area_rate:.3f} km^2 a^-1")
if glacier['obs_area_rate'] is not None:
    print(f"      [Observed: -{glacier['obs_area_rate']} km^2 a^-1, "
          f"{glacier['obs_area_note']}]")
else:
    print(f"      [Observed: (not set - "
          f"edit GLACIERS['{GLACIER_NAME}']['obs_area_rate'])]")
print(f"    Cumulative % lost: {area_pct_loss[-1]:.1f}%")

print(f"\n  VOLUME:")
print(f"    {YS}->{YE}: -{vol_loss_pct:.1f}% of {YS} volume")

print(f"\n  MASS BALANCE ({YS+1}-{YE}):")
print(f"    Model mean: {mass_balance.mean():.3f} m w.e. a^-1")
if glacier['obs_mb'] is not None:
    print(f"      [Observed: {glacier['obs_mb']} m w.e. a^-1, "
          f"{glacier['obs_mb_note']}]")
else:
    print(f"      [Observed: (not set - "
          f"edit GLACIERS['{GLACIER_NAME}']['obs_mb'])]")
print(f"    Method: B = dV x {ICE_DENSITY:.3f} / A_avg ")

print(f"\n  RUNOFF ({YS}-{YE}):")
if gr_vals is not None and len(gr_vals) > 0:
    print(f"    Mean glacier runoff: {gr_vals.mean():.1f} Mt a^-1")
    print(f"      (on-glacier melt + on-glacier liquid precip)")
    print(f"    Mean total runoff:   {tr_vals.mean():.1f} Mt a^-1")
    print(f"      (on- + off-glacier melt + liquid precip)")
else:
    print(f"    [Runoff unavailable - historical hydro run failed or empty]")

print(f"\n  SOURCES:")
print(f"    Area/volume: OGGM _spinup_historical  (level-5 preprocessed)")
print(f"    Calibration: Hugonnet et al. (2021), 2000-2020")
print(f"    Runoff: run_with_hydro, init_model_yr={YS}")


# ===========================================================================
# 10: FUTURE PROJECTION SUMMARY 
# ===========================================================================
# Reporting windows (per variable):

print(f"\n=== FUTURE PROJECTION SUMMARY ({label.upper()}) ===")
print(f"  Multi-GCM ensemble ({len(GCMS)} GCMs, CMIP6):")
for i, g in enumerate(GCMS):
    print(f'     {i+1}. {g}')
print()
print(f"  Reporting windows:")
print(f"    Volume / area:  2020-2100")
print(f"    Mass balance:   2021-2100  (V.diff)")
print(f"    Runoff:         {RUNOFF_YS}-{RUNOFF_YE}  "
      f"(Aguayo et al. 2025 - excludes partial endpoints)")
print()
print(f"  {'SSP':<10}{'V2020 (km^3)':>14}{'V2100 (km^3)':>15}"
      f"{'% remaining':>14}{'A2100 (km^2)':>15}"
      f"{'Peak runoff':>14}{'Peak year':>12}"
      f"{'R 2090-99 (Mt)':>16}{'R cum (Mt)':>13}")
print(f"  {'-'*10}{'-'*14}{'-'*15}{'-'*14}{'-'*15}"
      f"{'-'*14}{'-'*12}{'-'*16}{'-'*13}")

proj_rows = []
for ssp in SSPS:
    v = DS['volume'].sel(ssp=ssp).sum('rgi_id')
    a = DS['area'].sel(ssp=ssp).sum('rgi_id')
    
    r = runoff_trim.sel(ssp=ssp).sum('rgi_id') * 1e-9   # Mt yr^-1

    v_mean, v_std = v.mean('gcm'), v.std('gcm')
    a_mean = a.mean('gcm')
    r_mean = r.mean('gcm')

    
    v2020     = float(v_mean.sel(time=2020)) * 1e-9
    v2100     = float(v_mean.sel(time=2100)) * 1e-9
    v2100_std = float(v_std .sel(time=2100)) * 1e-9
    a2100     = float(a_mean.sel(time=2100)) * 1e-6
    pct       = v2100 / v2020 * 100 if v2020 > 0 else 0.0
    pct_std   = v2100_std / v2020 * 100 if v2020 > 0 else 0.0

    
    peak_yr   = int(r_mean.idxmax('time').item())
    peak_val  = float(r_mean.max('time'))

    
    r_2090_2099_mean = float(r_mean.sel(time=slice(2090, 2099)).mean())
    r_cum            = float(r_mean.sel(time=slice(RUNOFF_YS, RUNOFF_YE)).sum())
    r_2021           = float(r_mean.sel(time=RUNOFF_YS))
    r_2099           = float(r_mean.sel(time=RUNOFF_YE))
    r_change_pct     = (r_2099 / r_2021 * 100) if r_2021 > 0 else float('nan')

    print(f"  {SSP_LABELS[ssp]:<10}"
          f"{v2020:>14.4f}{v2100:>15.4f}{pct:>12.1f}±{pct_std:.0f}%"
          f"{a2100:>15.3f}{peak_val:>12.1f}Mt{peak_yr:>12d}"
          f"{r_2090_2099_mean:>16.1f}{r_cum:>13.0f}")

    
    dv = v.diff('time')
    B  = dv * ICE_DENSITY / A_RGI_TOT_M2
    B_mean = float(B.mean('gcm').mean('time'))

    v_2071_2100 = v_mean.sel(time=slice(2071, 2100)) * 1e-9
    a_2071_2100 = a_mean.sel(time=slice(2071, 2100)) * 1e-6

    proj_rows.append({
        'ssp':                          ssp,
        # State variables (2020-2100)
        'V_2020_km3':                   v2020,
        'V_2100_km3':                   v2100,
        'V_2100_km3_std':               v2100_std,
        'pct_remaining_2100':           pct,
        'pct_remaining_2100_std':       pct_std,
        'A_2100_km2':                   a2100,
        'V_2071_2100_mean_km3':         float(v_2071_2100.mean()),
        'A_2071_2100_mean_km2':         float(a_2071_2100.mean()),
        'century_mean_mb_mwe_per_yr':   B_mean,
        # Runoff (2021-2099, Aguayo window)
        'runoff_window':                f'{RUNOFF_YS}-{RUNOFF_YE}',
        'peak_runoff_Mt':               peak_val,
        'peak_runoff_year':             peak_yr,
        'runoff_2021_Mt':               r_2021,
        'runoff_2099_Mt':               r_2099,
        'runoff_2090_2099_mean_Mt':     r_2090_2099_mean,
        'runoff_cumulative_Mt':         r_cum,
        'runoff_pct_change_2099_vs_2021': r_change_pct,
    })

pd.DataFrame(proj_rows).to_csv(
    os.path.join(OUTDIR, f'{GLACIER_NAME}_projection_summary.csv'),
    index=False)


rows = []
for ssp in SSPS:
    v = DS['volume'].sel(ssp=ssp).sum('rgi_id')
    a = DS['area'].sel(ssp=ssp).sum('rgi_id')
    r = runoff_trim.sel(ssp=ssp).sum('rgi_id')   # kg yr^-1, 2021-2099

    runoff_years = set(r.time.values.tolist())
    for t in v.time.values:
        rec = {
            'ssp':              ssp,
            'year':             int(t),
            'volume_m3_mean':   float(v.sel(time=t).mean('gcm')),
            'volume_m3_std':    float(v.sel(time=t).std('gcm')),
            'area_m2_mean':     float(a.sel(time=t).mean('gcm')),
            'area_m2_std':      float(a.sel(time=t).std('gcm')),
        }
        if t in runoff_years:
            rec['runoff_kg_mean'] = float(r.sel(time=t).mean('gcm'))
            rec['runoff_kg_std']  = float(r.sel(time=t).std('gcm'))
        else:
            rec['runoff_kg_mean'] = float('nan')
            rec['runoff_kg_std']  = float('nan')
        rows.append(rec)
pd.DataFrame(rows).to_csv(
    os.path.join(OUTDIR, f'{GLACIER_NAME}_projection_annual.csv'),
    index=False)

# ===========================================================================
# 11: EXPORT SUMMARY
# ===========================================================================
EXPORTS = [
    ('Historical 6-panel figure',     f'{GLACIER_NAME}_historical_{YS}_{YE}.png'),
    ('Historical area+volume CSV',    f'{GLACIER_NAME}_area_volume_{YS}_{YE}.csv'),
    ('Historical mass balance CSV',   f'{GLACIER_NAME}_mass_balance_{YS+1}_{YE}.csv'),
    ('Historical runoff CSV',         f'{GLACIER_NAME}_runoff_{YS}_{YE}.csv'),
    ('Projection volume change plot', 'fig_volume_change.png'),
    ('Projection area change plot',   'fig_area_change.png'),
    ('Projection mass balance plot',  'fig_mass_balance.png'),
    ('Projection runoff plot',        'fig_runoff.png'),
    ('Projection thickness plot',     'fig_thickness.png'),
    ('Projection raw output (NetCDF)', f'{GLACIER_NAME}_all_runs.nc'),
    ('Projection annual CSV',          f'{GLACIER_NAME}_projection_annual.csv'),
    ('Projection summary CSV',         f'{GLACIER_NAME}_projection_summary.csv'),
]

print('\n' + '=' * 72)
print(f'EXPORT SUMMARY - {len(EXPORTS)} files in:')
print(f'   {OUTDIR}')
print('=' * 72)
for lbl, fname in EXPORTS:
    path = os.path.join(OUTDIR, fname)
    status = 'OK ' if os.path.exists(path) else 'MISSING'
    size = (f'{os.path.getsize(path)/1024:.1f} KB'
            if os.path.exists(path) else '')
    print(f'  [{status}] {lbl:32s}  {fname:42s}  {size}')
print('=' * 72)
print('\nTo download from the OGGM Hub:')
print(f'  1. In JupyterLab (left panel), open the "{GLACIER_NAME}_outputs" '
      f'folder - it sits next to this notebook.')
print('  2. Right-click each file -> "Download".')
print('\nFull paths for copy-paste:')
for _, fname in EXPORTS:
    print(f'   {os.path.join(OUTDIR, fname)}')